# Team B — Nsight Systems + Nsight Compute Profiling
**IBM HPML Uncertainty-Aware Inference Project — Systems Lead**

Run this notebook on a **GCP VM with an A100 (or L4)**.  
It does NOT run on Colab free tier (nsys/ncu require `perf_event_paranoid ≤ 1`).

### What this notebook does
1. Checks nsys/ncu availability and GPU permissions
2. Runs `nsys profile` for each PTQ config → `.nsys-rep` timeline files
3. Runs `ncu` for each config, targeting its dominant kernels → hardware counter CSVs
4. Calls `nsight_roofline.py` to generate three figures:
   - Hardware-counter Roofline (all configs + memory level curves)
   - Speed-of-Light SM% vs DRAM% bar chart
   - ncu AI vs PyTorch Profiler AI comparison
5. Prints the bound-classification table and saves results to Drive

### GCP setup (run once before this notebook)
```bash
# On the GCP VM as root:
sudo sh -c 'echo 0 > /proc/sys/kernel/perf_event_paranoid'

# Install nsight tools if not present (CUDA 12 already includes them):
apt-get install -y nsight-systems-cli nsight-compute

# Or install nvtx Python package for richer domain support:
pip install nvtx
```


## 0 · Environment Checks

In [ ]:
import subprocess, os, sys, shutil
from pathlib import Path

# ── GPU info ─────────────────────────────────────────────────────────────────
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# ── Check nsys ───────────────────────────────────────────────────────────────
NSYS_PATHS = [
    "nsys",
    "/usr/local/cuda/bin/nsys",
    "/usr/local/cuda-12/bin/nsys",
    "/opt/nvidia/nsight-systems/2023.4.1/bin/nsys",
]
NSYS_BIN = next((p for p in NSYS_PATHS if shutil.which(p) or Path(p).is_file()), None)
if NSYS_BIN:
    r = subprocess.run([NSYS_BIN, "--version"], capture_output=True, text=True)
    print("nsys:", r.stdout.strip().split("\n")[0])
else:
    print("⚠  nsys not found — install: apt-get install -y nsight-systems-cli")

# ── Check ncu ────────────────────────────────────────────────────────────────
NCU_PATHS = [
    "ncu",
    "/usr/local/cuda/bin/ncu",
    "/usr/local/cuda-12/bin/ncu",
    "/opt/nvidia/nsight-compute/2023.3.0/ncu",
]
NCU_BIN = next((p for p in NCU_PATHS if shutil.which(p) or Path(p).is_file()), None)
if NCU_BIN:
    r = subprocess.run([NCU_BIN, "--version"], capture_output=True, text=True)
    print("ncu :", r.stdout.strip().split("\n")[0])
else:
    print("⚠  ncu not found — install: apt-get install -y nsight-compute")

# ── perf_event_paranoid ──────────────────────────────────────────────────────
try:
    val = int(Path("/proc/sys/kernel/perf_event_paranoid").read_text())
    if val <= 1:
        print(f"\n✓ perf_event_paranoid = {val} (ncu hardware counters will work)")
    else:
        print(f"\n⚠  perf_event_paranoid = {val} — must be ≤ 1 for ncu")
        print("   Fix: sudo sh -c 'echo 0 > /proc/sys/kernel/perf_event_paranoid'")
except Exception as e:
    print(f"  Cannot read perf_event_paranoid: {e}")


In [ ]:
# ── Check NVTX availability ───────────────────────────────────────────────
try:
    import torch.cuda.nvtx as nvtx
    nvtx.range_push("test"); nvtx.range_pop()
    print("✓ torch.cuda.nvtx available")
except Exception as e:
    print(f"  torch.cuda.nvtx: {e}")

try:
    import nvtx as nvtx_pkg
    print(f"✓ nvtx package available (domain + color support)")
except ImportError:
    print("  nvtx package not installed — install: pip install nvtx")
    print("  (torch.cuda.nvtx will be used as fallback)")


## 1 · Paths and Configuration

In [ ]:
import os
from pathlib import Path

# ── Edit these to match your GCP environment ─────────────────────────────────
REPO_DIR      = Path("/home/user/uncertainty-aware-inference/TeamB")  # repo root
HF_TOKEN      = os.environ.get("HF_TOKEN", "")   # or set directly

NSYS_OUT_DIR  = REPO_DIR / "nsys_results"
NCU_OUT_DIR   = REPO_DIR / "ncu_results"
PROF_DIR      = REPO_DIR / "profiler_results"     # existing PyTorch Profiler JSONs
ROOFLINE_DIR  = REPO_DIR / "roofline_figures"

for d in [NSYS_OUT_DIR, NCU_OUT_DIR, ROOFLINE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Configs to profile ────────────────────────────────────────────────────────
# Start with one config to verify everything works, then uncomment --all
TARGET_CONFIG = "mistral-7b-gptq-int4"   # good first test: has clear custom kernels
# Set to None to profile all configs
# TARGET_CONFIG = None

print(f"Repo dir    : {REPO_DIR}")
print(f"nsys output : {NSYS_OUT_DIR}")
print(f"ncu output  : {NCU_OUT_DIR}")
print(f"Profile dir : {PROF_DIR}")
print(f"Roofline dir: {ROOFLINE_DIR}")


## 2 · Nsight Systems — Timeline Profiling

In [ ]:
# ── Run nsys for the target config (or all configs) ─────────────────────────
# This captures a system-wide timeline: CUDA kernels, CPU↔GPU sync,
# and NVTX range markers.  Output: .nsys-rep files you can open in the
# Nsight Systems GUI.
#
# The --nvtx flag in run_profiler.py emits the "profiling_region" NVTX range,
# which restricts nsys capture to just the profiled generate() steps
# (not model loading or warmup).

nsys_args = [
    sys.executable, str(REPO_DIR / "run_nsys.py"),
    "--output-dir", str(NSYS_OUT_DIR),
    "--repo-dir",   str(REPO_DIR),
    "--profile-steps", "3",
]
if TARGET_CONFIG:
    nsys_args += ["--config", TARGET_CONFIG]
else:
    nsys_args += ["--all"]
if HF_TOKEN:
    nsys_args += ["--hf-token", HF_TOKEN]

print("CMD:", " ".join(nsys_args))
print("(This will take a few minutes per config — nsys is overhead-lightweight)")
print()

result = subprocess.run(nsys_args, capture_output=False)  # stream output live
if result.returncode != 0:
    print(f"\n⚠  nsys run exited with code {result.returncode}")
else:
    print("\n✓ nsys profiling complete")

# List produced .nsys-rep files
rep_files = list(NSYS_OUT_DIR.glob("*.nsys-rep"))
print(f"\nProduced {len(rep_files)} .nsys-rep file(s):")
for f in rep_files:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# ── Export GPU trace stats from the .nsys-rep files ──────────────────────────
# run_nsys.py already does this automatically, but you can re-run it
# here with --stats-only to parse existing files without re-profiling.

nsys_parse_args = [
    sys.executable, str(REPO_DIR / "run_nsys.py"),
    "--stats-only",
    "--output-dir", str(NSYS_OUT_DIR),
]
if TARGET_CONFIG:
    nsys_parse_args += ["--config", TARGET_CONFIG]
else:
    nsys_parse_args += ["--all"]

subprocess.run(nsys_parse_args, capture_output=False)

# Display nsys summary JSONs
import json
for json_path in sorted(NSYS_OUT_DIR.glob("*_nsys_summary.json")):
    with open(json_path) as f:
        s = json.load(f)
    print(f"\n{s['config_key']}  —  GPU time (profiling region): {s['total_gpu_time_ms']:.1f} ms")
    for k in (s.get('top_kernels') or [])[:3]:
        print(f"  {k['name'][:55]:<55} {k['total_ms']:>8.1f} ms  ({k['pct']:.1f}%)")


## 3 · Nsight Compute — Hardware Counter Profiling

In [ ]:
# ── Verify perf_event_paranoid ───────────────────────────────────────────────
try:
    val = int(Path("/proc/sys/kernel/perf_event_paranoid").read_text())
    if val > 1:
        print(f"ERROR: perf_event_paranoid = {val}. Cannot run ncu.")
        print("Fix: sudo sh -c 'echo 0 > /proc/sys/kernel/perf_event_paranoid'")
        raise SystemExit(1)
    print(f"✓ perf_event_paranoid = {val}")
except FileNotFoundError:
    print("Not on Linux — skipping perf check")


In [ ]:
# ── Run ncu for the target config ────────────────────────────────────────────
# ncu targets only the 2 dominant kernels (identified from the existing
# profiler_results JSONs) to keep runtime manageable.
#
# Expect 5-15 minutes per config: ncu replays each targeted kernel multiple
# times to collect all hardware counter sections.
#
# Output: {config}_ncu_raw.csv  (metrics) + {config}_ncu.ncu-rep (GUI file)

ncu_args = [
    "sudo",   # ncu requires root on most Linux systems
    sys.executable, str(REPO_DIR / "run_ncu.py"),
    "--output-dir",    str(NCU_OUT_DIR),
    "--repo-dir",      str(REPO_DIR),
    "--prof-dir",      str(PROF_DIR),
    "--gpu",           "A100-80GB",   # project GPU
    "--profile-steps", "1",   # 1 step is enough — ncu uses kernel replay
]
if TARGET_CONFIG:
    ncu_args += ["--config", TARGET_CONFIG]
else:
    ncu_args += ["--all"]
if HF_TOKEN:
    ncu_args += ["--hf-token", HF_TOKEN]

print("CMD:", " ".join(ncu_args))
print("⏳ This will take 5-15 minutes per config (kernel replay overhead)...")
print()

result = subprocess.run(ncu_args, capture_output=False)
if result.returncode != 0:
    print(f"\n⚠  ncu exited with code {result.returncode}")
    print("If you see 'ERR_NVGPUCTRPERM': run 'sudo sh -c \'echo 0 > /proc/sys/kernel/perf_event_paranoid\''")
else:
    print("\n✓ ncu profiling complete")

csv_files = list(NCU_OUT_DIR.glob("*_ncu_raw.csv"))
print(f"\nProduced {len(csv_files)} CSV file(s): {[f.name for f in csv_files]}")


In [ ]:
# ── Parse ncu CSVs into metric JSONs ────────────────────────────────────────
# If ncu already ran and produced CSVs, you can re-parse without re-running ncu:

ncu_parse_args = [
    sys.executable, str(REPO_DIR / "run_ncu.py"),
    "--parse-only",
    "--output-dir", str(NCU_OUT_DIR),
    "--prof-dir",   str(PROF_DIR),
]
if TARGET_CONFIG:
    ncu_parse_args += ["--config", TARGET_CONFIG]
else:
    ncu_parse_args += ["--all"]

subprocess.run(ncu_parse_args, capture_output=False)

# Preview parsed metrics
for json_path in sorted(NCU_OUT_DIR.glob("*_ncu_metrics.json")):
    with open(json_path) as f:
        d = json.load(f)
    rl = d.get("roofline", {})
    print(f"\n{rl.get('config_key','?')}  (ridge_dram={rl.get('ridge_dram','?')} FLOPs/B)")
    for kname, r in rl.get("kernels", {}).items():
        print(f"  {kname[:45]:<45}  AI={r['ai_dram']:>8.1f}  "
              f"TFLOPS={r['achieved_tflops']:>6.3f}  SM={r['sm_pct']:>4.0f}%  "
              f"Mem={r['mem_pct']:>4.0f}%  → {r['bound']}")


## 4 · Generate Roofline Figures

In [ ]:
# Generates all three figures from nsight_roofline.py:
#   1. roofline_ncu_A100_80GB.png    — hardware-counter Roofline
#   2. speed_of_light_ncu.png        — SM% vs DRAM% bar chart
#   3. ai_comparison_ncu_vs_pytorch.png — ncu AI vs PyTorch Profiler AI

roofline_args = [
    sys.executable, str(REPO_DIR / "nsight_roofline.py"),
    "--ncu-dir",    str(NCU_OUT_DIR),
    "--prof-dir",   str(PROF_DIR),
    "--output-dir", str(ROOFLINE_DIR),
    "--gpu",        "A100-80GB",   # or "L4" if on a different instance
]

# If you also have Llama-2 ncu results from Week 5 cross-model profiling:
# roofline_args += ["--extra-ncu-dirs", "/path/to/TeamA/ncu_results:/path/to/TeamC/ncu_results"]

subprocess.run(roofline_args, capture_output=False)

# Display the figures inline
from IPython.display import Image, display
for fig_name in ["roofline_ncu_A100_80GB.png", "speed_of_light_ncu.png", "ai_comparison_ncu_vs_pytorch.png"]:
    p = ROOFLINE_DIR / fig_name
    if p.exists():
        print(f"\n{fig_name}:")
        display(Image(filename=str(p), width=900))


## 5 · Key Findings Summary

In [ ]:
# ── Print the bound classification table ─────────────────────────────────────
import csv
csv_path = ROOFLINE_DIR / "ncu_bound_classification.csv"
if csv_path.exists():
    import pandas as pd
    df = pd.read_csv(csv_path)
    print("\nBound Classification (from ncu hardware counters):")
    print(df.to_string(index=False))
    print()

    # Summary per quant type
    print("\nSummary by quantization method:")
    print(df.groupby("bound")["config_key"].apply(list).to_string())


In [ ]:
# ── Compare ncu AI vs PyTorch Profiler AI for each config ────────────────────
import json
from pathlib import Path

print("\nncu (hardware) vs PyTorch Profiler (estimated) Arithmetic Intensity:")
print(f"  {'Config':<30} {'PyTorch AI':>12} {'ncu AI (DRAM)':>15} {'ratio':>7} Note")
print(f"  {'-'*80}")

prof_summary = PROF_DIR / "profiler_summary.json"
if prof_summary.exists():
    with open(prof_summary) as f:
        pt = json.load(f)

    for config_key in pt:
        pt_ai = pt[config_key]["compute"]["arithmetic_intensity"]
        ncu_json = NCU_OUT_DIR / f"{config_key}_ncu_metrics.json"
        if not ncu_json.exists():
            continue
        with open(ncu_json) as f:
            ncu_d = json.load(f)
        rl = ncu_d.get("roofline", {})
        kernels = rl.get("kernels", {})
        if not kernels:
            continue
        # Take the max AI across targeted kernels (representative kernel)
        ncu_ai = max(r.get("ai_dram", 0) for r in kernels.values())
        ratio  = ncu_ai / pt_ai if pt_ai > 0 else float("inf")
        note   = ""
        if pt_ai == 0:
            note = "← PyTorch FLOPs counter returned 0 (custom kernel)"
        elif ratio > 3:
            note = "← PyTorch significantly underestimates AI"
        elif ratio < 0.5:
            note = "← PyTorch overestimates AI"
        print(f"  {config_key:<30} {pt_ai:>12.1f} {ncu_ai:>15.1f} {ratio:>7.2f}  {note}")


## 6 · Save Results to Drive (Optional)

In [ ]:
# Copy all ncu output and roofline figures to Google Drive for persistence.
# Only needed if this is a temporary GCP VM that will be shut down.

import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive/uai_results")
if DRIVE_ROOT.exists():
    dst_ncu      = DRIVE_ROOT / "TeamB" / "ncu_results"
    dst_nsys     = DRIVE_ROOT / "TeamB" / "nsys_results"
    dst_roofline = DRIVE_ROOT / "TeamB" / "roofline_figures"

    for src, dst in [(NCU_OUT_DIR, dst_ncu), (NSYS_OUT_DIR, dst_nsys), (ROOFLINE_DIR, dst_roofline)]:
        if src.exists():
            dst.mkdir(parents=True, exist_ok=True)
            for f in src.iterdir():
                if f.suffix in (".json", ".csv", ".png"):  # skip large .nsys-rep / .ncu-rep
                    shutil.copy2(f, dst / f.name)
    print("Results copied to Drive.")
    print("Note: .nsys-rep and .ncu-rep GUI files are large — download them separately if needed.")
else:
    print("Drive not mounted — results remain on the GCP VM at:")
    print(f"  {NCU_OUT_DIR}")
    print(f"  {NSYS_OUT_DIR}")
    print(f"  {ROOFLINE_DIR}")
